In [2]:
# Import necessary libraries and modules
from datetime import date
from classes.payment import Payment
from classes.currency import Currency
from classes.direction import PaymentDirection, TradeDirection

In [ ]:
# Test case 1: Create a Payment object and verify attributes
payment = Payment(
    date=date(2023, 10, 1),
    currency=Currency('USD'),
    amount=1000.0,
    direction=PaymentDirection.BUY,
)
assert payment.date == date(2023, 10, 1)
assert payment.currency == Currency('USD')
assert payment.amount == 1000.0
assert payment.direction == PaymentDirection('RECEIVE')
#assert payment.amount_usd == 1000.0
print(payment)

In [ ]:
# Test case: Create an FXTrade object and verify attributes
from classes.FXTrade import FXTrade
from classes.currency_pair import CurrencyPair
#from classes.FXrate import FXRate

fx_trade = FXTrade(
    trade_date=date(2023, 10, 1),
    giv_amount=1000000.0,
    currency_pair=CurrencyPair.from_string('USD','EUR'),
    rate=1.1,
    giv_direction=TradeDirection.SELL,
    value_date=date(2023, 10, 3)
)

assert fx_trade.trade_date == date(2023, 10, 1)
assert fx_trade.giv_amount == 1000000.0
#assert fx_trade.currency_pair.get_domestic_currency == Currency.USD
#assert fx_trade.currency_pair.get_foreign_currency == Currency.EUR
assert fx_trade.rate == 1.1
assert fx_trade.direction == TradeDirection.SELL
print(fx_trade)

In [2]:
# Test case: Create an FXSwap object and verify attributes
from classes.FXSwap import FXSwap
from classes.FXTrade import FXTrade
from classes.currency_pair import CurrencyPair
from classes.direction import TradeDirection
from datetime import date

fx_trade_near = FXTrade(
    trade_date=date(2023, 10, 1),
    giv_amount=1000000.0,
    currency_pair=CurrencyPair.from_string('USD', 'EUR'),
    rate=1.1,
    giv_direction=TradeDirection.BUY,
    value_date=date(2023, 10, 3)
)

fx_trade_far = FXTrade(
    trade_date=date(2023, 10, 1),
    giv_amount=1000000.0,
    currency_pair=CurrencyPair.from_string('USD', 'EUR'),
    rate=1.2,
    giv_direction=TradeDirection.SELL,
    value_date=date(2023, 10, 10)
)

fx_swap = FXSwap(fx_trade_near=fx_trade_near, fx_trade_far=fx_trade_far)

assert fx_swap.fx_trade_near == fx_trade_near
assert fx_swap.fx_trade_far == fx_trade_far
assert fx_swap.direction == TradeDirection.BUY_SELL
assert fx_swap.price == fx_trade_far.rate - fx_trade_near.rate
print(fx_swap)

In [9]:
from collections import defaultdict
from classes.currency_pair import CurrencyPair
from classes.FXTrade import FXTrade

# Example: Calculate positions by value date from multiple FX trades

# List of FX trades
fx_trades = [
    FXTrade(
        trade_date=date(2023, 10, 1),
        giv_amount=1000000.0,
        currency_pair=CurrencyPair.from_string('USD', 'EUR'),
        rate=1.1,
        giv_direction=TradeDirection.BUY,
        value_date=date(2023, 10, 3)
    ),
    FXTrade(
        trade_date=date(2023, 10, 2),
        giv_amount=500000.0,
        currency_pair=CurrencyPair.from_string('USD', 'JPY'),
        rate=110.0,
        giv_direction=TradeDirection.SELL,
        value_date=date(2023, 10, 4)
    ),
    FXTrade(
        trade_date=date(2023, 10, 3),
        giv_amount=750000.0,
        currency_pair=CurrencyPair.from_string('EUR', 'USD'),
        rate=1.2,
        giv_direction=TradeDirection.BUY,
        value_date=date(2023, 10, 5)
    )
]

# Calculate positions grouped by value date
positions_by_date = defaultdict(lambda: defaultdict(float))

for trade in fx_trades:
    value_date = trade.value_date
    if trade.giv_direction == TradeDirection.BUY:
        positions_by_date[value_date][trade.currency_pair.get_domestic_currency()] += trade.giv_amount
        positions_by_date[value_date][trade.currency_pair.get_foreign_currency()] -= trade.giv_amount * trade.rate
    elif trade.giv_direction == TradeDirection.SELL:
        positions_by_date[value_date][trade.currency_pair.get_domestic_currency()] -= trade.giv_amount
        positions_by_date[value_date][trade.currency_pair.get_foreign_currency()] += trade.giv_amount * trade.rate

# Print positions grouped by value date
for value_date, positions in positions_by_date.items():
    print(f"Value Date: {value_date}")
    for currency, position in positions.items():
        print(f"  {currency}: {position}")

AttributeError: 'FXTrade' object has no attribute 'value_date'